In [8]:
# You can use this section to suppress warnings generated by your code:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore') 
import os 
import getpass
from langchain_groq import ChatGroq 
from langchain_community.document_loaders import TextLoader 
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [2]:
def set_if_undefined(var:str): 
    if os.environ.get(var): 
        return 
    os.environ[var] = getpass.getpass(var) 
set_if_undefined("GROQ_API_KEY")    

GROQ_API_KEY ········


In [3]:
#Build a llm model 
def llm_model(model_id:str): 
    llm = ChatGroq(
        model = model_id, 
        temperature= 0.5, 
        max_tokens = 256, 
        model_kwargs = {
            "top_p":0.8
        }
    )
    return llm

In [4]:
groq_model = llm_model("llama-3.3-70b-versatile")
# Testing whether model is created and working 
response = groq_model.invoke("What is your name?")
response

AIMessage(content='I\'m an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 40, 'total_tokens': 63, 'completion_time': 0.031844237, 'completion_tokens_details': None, 'prompt_time': 0.008220597, 'prompt_tokens_details': None, 'queue_time': 0.069133302, 'total_time': 0.040064834}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_d42c28f9ce', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e8e82-518e-7ca0-a15a-142f85cffb05-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 40, 'output_tokens': 23, 'total_tokens': 63})

In [5]:
!wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/d_ahNwb1L2duIxBR6RD63Q/state-of-the-union.txt"

--2026-06-03 10:23:02--  https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/d_ahNwb1L2duIxBR6RD63Q/state-of-the-union.txt
Resolving cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)... 198.23.119.245
Connecting to cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud (cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud)|198.23.119.245|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 39027 (38K) [text/plain]
Saving to: ‘state-of-the-union.txt’

state-of-the-union. 100%[===================>]  38.11K  --.-KB/s    in 0.1s    

2026-06-03 10:23:02 (336 KB/s) - ‘state-of-the-union.txt’ saved [39027/39027]



In [10]:
#Using Text loader 
loader = TextLoader("state-of-the-union.txt")
data = loader.load() 
content = data[0].page_content

In [7]:
#Prompt template
template = """According to the document content here 
            {content},
            answer this question 
            {question}.
            Do not try to make up the answer.
                
            YOUR RESPONSE:
"""
prompt_template = PromptTemplate.from_template(template) 
prompt_template

PromptTemplate(input_variables=['content', 'question'], input_types={}, partial_variables={}, template='According to the document content here \n            {content},\n            answer this question \n            {question}.\n            Do not try to make up the answer.\n\n            YOUR RESPONSE:\n')

In [13]:
chain = prompt_template| groq_model | StrOutputParser() 
query = "It is in which year of our nation?."
response = chain.invoke({"content":content, "question": query})
response

'Our 245th year as a nation.'